<a href="https://colab.research.google.com/github/subinyy/IS_NetShield/blob/main/awsEC2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# url_features.py


In [ ]:
url_features_code = '''
import ipaddress, re, urllib.parse as up
import numpy as np, pandas as pd

SUSPICIOUS_KEYWORDS = [
    "login","signin","verify","update","secure","account","bank","bonus","gift",
    "wallet","confirm","password","auth","invoice","payment","recover","reset",
    "apple","paypal","amazon","microsoft","office","netflix","facebook","meta",
    "telegram","steam","roblox","crypto","airdrop",
]
SUSPICIOUS_TLDS = {"top","xyz","click","work","shop","gq","fit","rest",
                   "country","stream","mom","zip","mov","cam","info","live"}
URL_SHORTENERS = ["bit.ly","tinyurl","cutt.ly","t.co","rb.gy"]
FEATURES = [
    "url_length","host_length","path_length","query_length","digit_count",
    "special_char_count","dot_count","hyphen_count","slash_count","subdomain_count",
    "param_count","has_ip_host","has_https_token","has_login_keyword",
    "suspicious_keyword_hits","entropy","is_suspicious_tld","uses_shortener_hint",
]

def normalize_url(url: str):
    if not isinstance(url, str): return None
    url = url.strip()
    if not url: return None
    if not re.match(r"^[a-zA-Z][a-zA-Z0-9+\\-.]*://", url):
        url = "https://" + url
    try:
        p = up.urlsplit(url)
        scheme = p.scheme.lower()
        if scheme not in {"http","https"}: return None
        hostname = p.hostname.lower() if p.hostname else None
        if not hostname: return None
        try: port = p.port
        except: return None
        if (scheme=="http" and port==80) or (scheme=="https" and port==443): port = None
        netloc = hostname + (f":{port}" if port else "")
        path  = re.sub(r"/+", "/", p.path or "/")
        query = up.urlencode(sorted(up.parse_qsl(p.query, keep_blank_values=True)))
        return up.urlunsplit((scheme, netloc, path, query, ""))
    except: return None

def _entropy(s):
    if not s: return 0.0
    counts = np.array(list(pd.Series(list(s)).value_counts().values), dtype=float)
    probs = counts / counts.sum()
    return float(-(probs * np.log2(probs)).sum())

def _is_ip(host):
    try: ipaddress.ip_address(host); return 1
    except: return 0

def extract_features(url: str) -> dict:
    norm = normalize_url(url)
    if norm is None: return {f: 0 for f in FEATURES}
    try: p = up.urlsplit(norm)
    except: return {f: 0 for f in FEATURES}
    host = p.hostname or ""; path = p.path or ""; query = p.query or ""
    fl = norm.lower(); hl = host.lower(); pq = (path+"?"+query).lower()
    return {
        "url_length": len(norm), "host_length": len(host),
        "path_length": len(path), "query_length": len(query),
        "digit_count": sum(c.isdigit() for c in norm),
        "special_char_count": sum(c in ".@-_=+%?&#!$," for c in norm),
        "dot_count": norm.count("."), "hyphen_count": norm.count("-"),
        "slash_count": norm.count("/"),
        "subdomain_count": max(len(host.split("."))-2,0) if host else 0,
        "param_count": len(up.parse_qsl(query,keep_blank_values=True)) if query else 0,
        "has_ip_host": _is_ip(host),
        "has_https_token": int("https" in hl or "https" in pq),
        "has_login_keyword": int(any(k in fl for k in ["login","signin","verify","password","account"])),
        "suspicious_keyword_hits": sum(k in fl for k in SUSPICIOUS_KEYWORDS),
        "entropy": round(_entropy(norm),4),
        "is_suspicious_tld": int(hl.rsplit(".",1)[-1] in SUSPICIOUS_TLDS if "." in hl else 0),
        "uses_shortener_hint": int(any(x in hl for x in URL_SHORTENERS)),
    }

def extract_features_batch(urls):
    return pd.DataFrame([extract_features(u) for u in urls])[FEATURES]
'''

with open("url_features.py","w") as f:
    f.write(url_features_code)
print("url_features.py 생성 완료")

from url_features import extract_features, normalize_url
t = "http://login-paypal-verify.top/secure/confirm?s=abc"
feat = extract_features(t)
print(f"테스트: entropy={feat['entropy']}  kw_hits={feat['suspicious_keyword_hits']}  susp_tld={feat['is_suspicious_tld']}")

# aws EC2 배포용

In [ ]:
import nest_asyncio, uvicorn, threading, pickle, json, time, os
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field
from typing import Optional, List
import pandas as pd

# ==========================================
# 1. 비동기 환경 설정
# ==========================================
nest_asyncio.apply()

# ==========================================
# 2. 머신러닝 모델 및 메타데이터 로드
# ==========================================
with open("models/url_detector_best.pkl", "rb") as f:
    _model = pickle.load(f)
_meta = json.load(open("models/metrics.json"))
print(f" 모델 로드 성공: {_meta['best_model']} (Dataset Ver: {_meta['dataset_version']})")

THRESHOLD = 0.5

# ==========================================
# 3. FastAPI 인스턴스 및 메타데이터 정의
# ==========================================
tags_metadata = [
    {
        "name": " 시스템 및 모델 정보",
        "description": "서버의 가동 상태와 탐지 모델의 세부 메타데이터(버전, 사용된 피처 등)를 확인합니다.",
    },
    {
        "name": " 유해 URL 실시간 분석",
        "description": "사용자가 입력한 URL의 특징(Feature)을 분석하고 ML 모델을 통해 위험도를 실시간 예측합니다.",
    },
    {
        "name": " UI",
        "description": "IS_NetShield 대시보드 UI를 제공합니다.",
    },
]

app = FastAPI(
    title=" 유해 URL 실시간 탐지 API 시스템",
    description="""
 본 API 서비스는 입력된 URL의 구조적/텍스트적 피처를 실시간으로 분석하여 피싱 및 악성 웹사이트 여부를 판별합니다.

- **Single Predict**: 개별 URL 분석 및 세부 피처 정밀 확인
- **Batch Predict**: 대량의 URL을 하나의 요청으로 묶어 초고속 일괄 분석 (Vectorization 최적화 적용)
    """,
    version="2.0.0",
    openapi_tags=tags_metadata
)

# CORS 미들웨어 설정
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

# ==========================================
# 4. Pydantic 데이터 모델 정의
# ==========================================
class URLRequest(BaseModel):
    url: str = Field(..., example="www.secure-login-update.com", description="검사할 단일 웹사이트 URL 주소")
    threshold: Optional[float] = Field(None, description="악성 판정 임계값 (지정하지 않으면 서버 기본값 0.5 사용)")

class BatchURLRequest(BaseModel):
    urls: List[str] = Field(..., example=["http://example.com", "phishing-site.net", "naver.com"], description="일괄 검사할 URL 주소 리스트")
    threshold: Optional[float] = Field(None, description="악성 판정 임계값 (지정하지 않으면 서버 기본값 0.5 사용)")

class URLPredictionResult(BaseModel):
    url: str = Field(..., description="검사된 원본 또는 보정된 URL")
    normalized_url: str = Field(..., description="정규화(Normalization) 처리된 URL")
    is_malicious: bool = Field(..., description="유해/악성 여부 결과 (True: 악성, False: 안전)")
    probability: float = Field(..., description="모델이 판단한 악성 사이트일 확률 (0.0 ~ 1.0)")
    confidence: str = Field(..., description="예측 결과의 신뢰도 수준 (HIGH, MEDIUM, LOW)")
    risk_level: str = Field(..., description="최종 위험 등급 (DANGEROUS, SUSPICIOUS, SAFE, ERROR)")
    features: Optional[dict] = Field(None, description="URL에서 추출된 텍스트 및 구조적 피처 데이터 딕셔너리")
    threshold_used: float = Field(..., description="이번 예측에 적용된 임계값(Threshold)")
    error: Optional[str] = Field(None, description="URL 처리 중 에러 발생 시 에러 메시지 기술 (정상 처리 시 null)")

class BatchSummary(BaseModel):
    total: int = Field(..., description="총 요청된 URL 수")
    malicious: int = Field(..., description="유해 사이트로 판정된 수")
    safe: int = Field(..., description="안전한 사이트로 판정된 수")
    error_count: int = Field(..., description="피처 추출 등 처리에 실패한 URL 수")
    malicious_ratio: float = Field(..., description="성공적으로 검사된 URL 중 유해 사이트의 비율 (0.0 ~ 1.0)")

class BatchDetectionResponse(BaseModel):
    results: List[URLPredictionResult] = Field(..., description="각 URL별 상세 검사 결과 리스트")
    summary: BatchSummary = Field(..., description="이번 대량 검사의 통계 요약 데이터")
    elapsed_ms: float = Field(..., description="서버 내부 연산에 소요된 대기 시간 (밀리초 단위)")


# ==========================================
# 5. 내부 비즈니스 및 유틸리티 로직 함수
# ==========================================
def sanitize_url(url: str) -> str:
    url = url.strip()
    if not url.startswith(("http://", "https://")):
        return f"https://{url}"
    return url

def _parse_prediction_details(prob: float, thr: float) -> tuple:
    is_mal = prob >= thr
    if prob >= 0.85:   return is_mal, "HIGH", "DANGEROUS"
    elif prob >= thr:  return is_mal, "MEDIUM", "SUSPICIOUS"
    elif prob >= 0.3:  return is_mal, "LOW", "SUSPICIOUS"
    else:              return is_mal, "HIGH", "SAFE"


# ==========================================
# 6. API 엔드포인트 선언
# ==========================================
@app.get("/health", tags=["⚙️ 시스템 및 모델 정보"], summary="서버 및 모델 상태 점검")
def health():
    return {"status": "ok", "model": _meta["best_model"], "dataset": _meta["dataset_version"]}

@app.get("/model/info", tags=["⚙️ 시스템 및 모델 정보"], summary="모델 상세 메타데이터 조회")
def info():
    from url_features import FEATURES
    return {
        "model": _meta["best_model"],
        "dataset_version": _meta["dataset_version"],
        "features": FEATURES,
        "threshold": THRESHOLD,
        "test_auc": _meta.get(f"{_meta['best_model'].lower()[:3]}_test_auc")
    }

@app.post("/predict", tags=["🔍 유해 URL 실시간 분석"], summary="단일 URL 위험도 검사", response_model=URLPredictionResult)
def predict(req: URLRequest):
    from url_features import extract_features, normalize_url, FEATURES

    thr = req.threshold or THRESHOLD
    target_url = sanitize_url(req.url)
    try:
        feat = extract_features(target_url)
        df = pd.DataFrame([feat])[FEATURES].astype(float)
        prob = float(_model.predict_proba(df)[0, 1])
        is_mal, conf_lv, risk = _parse_prediction_details(prob, thr)
        return {
            "url": target_url, "normalized_url": normalize_url(target_url), "is_malicious": is_mal,
            "probability": round(prob, 4), "confidence": conf_lv, "risk_level": risk,
            "features": feat, "threshold_used": thr, "error": None
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"URL 분석 중 오류가 발생했습니다: {str(e)}")

@app.post("/predict/batch", tags=["🔍 유해 URL 실시간 분석"], summary="복수 URL 일괄 고성능 검사", response_model=BatchDetectionResponse)
def predict_batch(req: BatchURLRequest):
    from url_features import extract_features, normalize_url, FEATURES

    if not req.urls:
        raise HTTPException(status_code=400, detail="요청된 URL 배열이 비어 있습니다.")
    thr = req.threshold or THRESHOLD
    t0 = time.time()

    sanitized_urls = [sanitize_url(u) for u in req.urls]
    features_list = []
    valid_indices = []
    results = [None] * len(sanitized_urls)

    for idx, url in enumerate(sanitized_urls):
        try:
            feat = extract_features(url)
            features_list.append(feat)
            valid_indices.append(idx)
        except Exception as e:
            results[idx] = {
                "url": url, "normalized_url": url, "is_malicious": False,
                "probability": 0.0, "confidence": "UNKNOWN", "risk_level": "ERROR",
                "features": None, "threshold_used": thr, "error": f"Feature extraction failed: {str(e)}"
            }

    if features_list:
        batch_df = pd.DataFrame(features_list)[FEATURES].astype(float)
        probabilities = _model.predict_proba(batch_df)[:, 1]
        for v_idx, prob in zip(valid_indices, probabilities):
            prob_val = float(prob)
            url = sanitized_urls[v_idx]
            is_mal, conf_lv, risk = _parse_prediction_details(prob_val, thr)
            current_feat_idx = valid_indices.index(v_idx)
            results[v_idx] = {
                "url": url, "normalized_url": normalize_url(url), "is_malicious": is_mal,
                "probability": round(prob_val, 4), "confidence": conf_lv, "risk_level": risk,
                "features": features_list[current_feat_idx], "threshold_used": thr, "error": None
            }

    n_mal = sum(1 for r in results if r and r["is_malicious"])
    n_err = sum(1 for r in results if r and r["error"] is not None)
    return {
        "results": results,
        "summary": {
            "total": len(results), "malicious": n_mal, "safe": len(results) - n_mal - n_err, "error_count": n_err,
            "malicious_ratio": round(n_mal / (len(results) - n_err), 4) if (len(results) - n_err) > 0 else 0.0
        },
        "elapsed_ms": round((time.time() - t0) * 1000, 1)
    }

# ==========================================
# 7. UI 서빙 엔드포인트 (추가)
# ==========================================
@app.get("/", response_class=HTMLResponse, tags=["🖥️ UI"], summary="IS_NetShield 대시보드")
def serve_ui():
    html_path = "netshield.html"
    if os.path.exists(html_path):
        with open(html_path, "r", encoding="utf-8") as f:
            return f.read()
    return HTMLResponse("<h1>netshield.html 파일을 서버에 올려주세요</h1>", status_code=404)

# ==========================================
# 8. 웹서버 구동부 (AWS EC2 배포용)
# ==========================================
if __name__ == "__main__":
    print("\n 유해 URL 탐지 API 서버 가동을 시작합니다...")
    uvicorn.run(app, host="0.0.0.0", port=8000)